In [ ]:
"""
Zero Trust Security Intelligence Dashboard v2
Loads your actual generated graphs + Excel data directly.
Usage: python3 zero_trust_dashboard_v2.py
Place this file alongside the 'outputs/' folder.
"""

import os, sys, time, threading
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from tkinter.font import Font
from PIL import Image, ImageTk
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import matplotlib.patches as mpatches
import numpy as np
from openpyxl import load_workbook

# ─── THEME ───────────────────────────────────────────────────────────────────
BG      = "#0D1117"
PANEL   = "#161B22"
CARD    = "#1C2333"
BORDER  = "#30363D"
ACCENT  = "#58A6FF"
GREEN   = "#3FB950"
RED     = "#F85149"
ORANGE  = "#D29922"
PURPLE  = "#BC8CFF"
CYAN    = "#39C5CF"
TEXT    = "#E6EDF3"
SUBTEXT = "#8B949E"
TRAD_C  = "#C0392B"
ZT_C    = "#1A5276"

GRAPH_FILES = {
    "Attack Detection Rate":        "graph1_attack_detection_rate.png",
    "Unauthorized Access Prev.":    "graph2_unauthorized_access_prevention.png",
    "Authentication Latency":       "graph3_authentication_latency.png",
    "Network Overhead":             "graph4_network_overhead.png",
    "Lateral Movement":             "graph5_lateral_movement.png",
    "Confusion Matrix":             "graph6_confusion_matrix.png",
    "Feature Importance":           "graph7_feature_importance.png",
    "Anomaly Score Distribution":   "graph8_anomaly_score_distribution.png",
    "Security Metrics Comparison":  "graph9_security_metrics_comparison.png",
    "Attack Class Distribution":    "graph10_attack_class_distribution.png",
    "Micro-Segmentation":           "graph11_micro_segmentation.png",
    "CPU Usage":                    "graph12_cpu_usage.png",
    "Class-wise Precision":         "graph13_classwise_precision.png",
    "Class-wise Recall":            "graph14_classwise_recall.png",
    "Class-wise F1-Score":          "graph15_classwise_f1.png",
    "ZT: All Metrics per Class":    "graph16_zt_classwise_all_metrics.png",
    "Traditional: All Metrics":     "graph17_trad_classwise_all_metrics.png",
    "Performance Metrics Line":     "graph18_performance_metrics_line.png",
}

# ─── HELPERS ─────────────────────────────────────────────────────────────────
def find_outputs_dir():
    """Search for 'outputs' folder relative to script location."""
    candidates = [
        os.path.join(os.path.dirname(os.path.abspath(__file__)), "outputs"),
        os.path.join(os.getcwd(), "outputs"),
        "outputs",
    ]
    for c in candidates:
        if os.path.isdir(c):
            return c
    return None

def load_excel_sheet(wb, sheet_name):
    if sheet_name not in wb.sheetnames:
        return [], []
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    if not rows:
        return [], []
    headers = [str(h) if h else "" for h in rows[0]]
    data = [tuple(str(c) if c is not None else "" for c in r) for r in rows[1:]]
    return headers, data

# ─── MAIN APP ────────────────────────────────────────────────────────────────
class ZeroTrustDashboard(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("🛡 Zero Trust Security Intelligence Dashboard")
        self.geometry("1560x920")
        self.minsize(1280, 780)
        self.configure(bg=BG)
        self.outputs_dir = find_outputs_dir()
        self.wb = None
        self._photo_cache = {}
        self._setup_styles()
        self._build_ui()
        self.after(100, self._load_data)

    def _setup_styles(self):
        s = ttk.Style(self)
        s.theme_use("clam")
        s.configure("TNotebook", background=BG, borderwidth=0)
        s.configure("TNotebook.Tab", background=CARD, foreground=SUBTEXT,
                    padding=[16,8], font=("Courier New",10,"bold"), borderwidth=0)
        s.map("TNotebook.Tab",
              background=[("selected", PANEL)], foreground=[("selected", ACCENT)])
        s.configure("Dark.Treeview", background=CARD, foreground=TEXT,
                    fieldbackground=CARD, rowheight=28, font=("Courier New",10))
        s.configure("Dark.Treeview.Heading", background=PANEL, foreground=ACCENT,
                    font=("Courier New",10,"bold"), relief="flat")
        s.map("Dark.Treeview",
              background=[("selected", ACCENT)], foreground=[("selected", BG)])
        s.configure("Horizontal.TProgressbar",
                    troughcolor=CARD, background=ACCENT,
                    darkcolor=ACCENT, lightcolor=ACCENT, bordercolor=BORDER)

    # ─── UI SKELETON ─────────────────────────────────────────────────────────
    def _build_ui(self):
        # TOP BAR
        top = tk.Frame(self, bg=PANEL, height=58)
        top.pack(fill="x"); top.pack_propagate(False)
        tk.Label(top, text="🛡  ZERO TRUST SECURITY INTELLIGENCE DASHBOARD",
                 bg=PANEL, fg=ACCENT, font=("Courier New",15,"bold")).pack(side="left", padx=22, pady=12)
        self.status_lbl = tk.Label(top, text="● LOADING", bg=PANEL, fg=ORANGE,
                                    font=("Courier New",10,"bold"))
        self.status_lbl.pack(side="right", padx=20)
        self.clock_lbl = tk.Label(top, text="", bg=PANEL, fg=SUBTEXT,
                                   font=("Courier New",10))
        self.clock_lbl.pack(side="right", padx=10)
        self._tick()

        # DIR BAR
        dir_bar = tk.Frame(self, bg=CARD, height=32)
        dir_bar.pack(fill="x"); dir_bar.pack_propagate(False)
        self.dir_var = tk.StringVar(value=self.outputs_dir or "outputs/ folder not found")
        tk.Label(dir_bar, text="📁 outputs:", bg=CARD, fg=SUBTEXT,
                 font=("Courier New",9)).pack(side="left", padx=10, pady=6)
        tk.Label(dir_bar, textvariable=self.dir_var, bg=CARD, fg=GREEN,
                 font=("Courier New",9)).pack(side="left")
        tk.Button(dir_bar, text="Browse…", bg=CARD, fg=ACCENT, relief="flat",
                  font=("Courier New",9,"bold"), cursor="hand2",
                  activebackground=BORDER, activeforeground=ACCENT,
                  command=self._browse_dir).pack(side="right", padx=12)

        # KPI STRIP
        kpi_row = tk.Frame(self, bg=BG)
        kpi_row.pack(fill="x", padx=14, pady=(6,4))
        self.kpi = {}
        kpi_defs = [
            ("ZT Accuracy",      "—", ACCENT),
            ("ZT Precision",     "—", GREEN),
            ("ZT Recall",        "—", CYAN),
            ("ZT F1-Score",      "—", PURPLE),
            ("Trad Accuracy",    "—", ORANGE),
            ("Trad F1-Score",    "—", RED),
            ("Anomaly Rate",     "—", "#FF6B6B"),
            ("Total Samples",    "—", SUBTEXT),
        ]
        for i,(label,val,color) in enumerate(kpi_defs):
            card = tk.Frame(kpi_row, bg=CARD, highlightbackground=BORDER, highlightthickness=1)
            card.grid(row=0, column=i, padx=4, sticky="ew")
            kpi_row.columnconfigure(i, weight=1)
            tk.Label(card, text=label.upper(), bg=CARD, fg=SUBTEXT,
                     font=("Courier New",7,"bold")).pack(pady=(7,1))
            var = tk.StringVar(value=val)
            tk.Label(card, textvariable=var, bg=CARD, fg=color,
                     font=("Courier New",16,"bold")).pack(pady=(0,7))
            self.kpi[label] = var

        # NOTEBOOK
        self.nb = ttk.Notebook(self)
        self.nb.pack(fill="both", expand=True, padx=14, pady=(2,10))

        self._build_tab_gallery()
        self._build_tab_overview()
        self._build_tab_classwise()
        self._build_tab_network()
        self._build_tab_tables()
        self._build_tab_mitre()

    # ─── TAB: IMAGE GALLERY ──────────────────────────────────────────────────
    def _build_tab_gallery(self):
        f = tk.Frame(self.nb, bg=BG)
        self.nb.add(f, text="🖼  ALL GRAPHS")

        ctrl = tk.Frame(f, bg=BG)
        ctrl.pack(fill="x", padx=10, pady=6)
        tk.Label(ctrl, text="Graph:", bg=BG, fg=TEXT,
                 font=("Courier New",10,"bold")).pack(side="left", padx=6)
        self.graph_sel = tk.StringVar(value=list(GRAPH_FILES.keys())[0])
        om = tk.OptionMenu(ctrl, self.graph_sel, *GRAPH_FILES.keys(),
                           command=self._show_graph)
        om.config(bg=CARD, fg=ACCENT, activebackground=PANEL, activeforeground=TEXT,
                  highlightbackground=BORDER, font=("Courier New",10,"bold"),
                  relief="flat", width=34)
        om["menu"].config(bg=CARD, fg=TEXT, activebackground=ACCENT,
                          activeforeground=BG, font=("Courier New",9))
        om.pack(side="left", padx=8)

        # Nav buttons
        tk.Button(ctrl, text="◀ Prev", bg=CARD, fg=ACCENT, relief="flat",
                  font=("Courier New",10,"bold"), cursor="hand2",
                  activebackground=BORDER, command=self._prev_graph).pack(side="left", padx=4)
        tk.Button(ctrl, text="Next ▶", bg=CARD, fg=ACCENT, relief="flat",
                  font=("Courier New",10,"bold"), cursor="hand2",
                  activebackground=BORDER, command=self._next_graph).pack(side="left", padx=4)

        self.graph_name_lbl = tk.Label(ctrl, text="", bg=BG, fg=SUBTEXT,
                                        font=("Courier New",9))
        self.graph_name_lbl.pack(side="right", padx=10)

        # Image canvas with scrollbar
        img_frame = tk.Frame(f, bg=BG)
        img_frame.pack(fill="both", expand=True, padx=8, pady=4)
        self.img_canvas = tk.Canvas(img_frame, bg=CARD, highlightthickness=0)
        vsb = ttk.Scrollbar(img_frame, orient="vertical", command=self.img_canvas.yview)
        hsb = ttk.Scrollbar(img_frame, orient="horizontal", command=self.img_canvas.xview)
        self.img_canvas.configure(yscrollcommand=vsb.set, xscrollcommand=hsb.set)
        vsb.pack(side="right", fill="y")
        hsb.pack(side="bottom", fill="x")
        self.img_canvas.pack(fill="both", expand=True)
        self.img_canvas.bind("<Configure>", self._on_canvas_resize)
        self._current_graph_keys = list(GRAPH_FILES.keys())
        self._current_graph_idx = 0

    def _show_graph(self, name=None):
        if name is None:
            name = self.graph_sel.get()
        if not self.outputs_dir:
            return
        fname = GRAPH_FILES.get(name, "")
        fpath = os.path.join(self.outputs_dir, fname)
        if not os.path.exists(fpath):
            self.graph_name_lbl.config(text=f"NOT FOUND: {fname}", fg=RED)
            return
        self.graph_name_lbl.config(text=fname, fg=SUBTEXT)
        self._render_image(fpath)

    def _render_image(self, fpath):
        try:
            img = Image.open(fpath)
            cw = max(self.img_canvas.winfo_width(), 800)
            ch = max(self.img_canvas.winfo_height(), 500)
            scale = min(cw / img.width, ch / img.height, 1.0)
            nw, nh = int(img.width * scale), int(img.height * scale)
            img_resized = img.resize((nw, nh), Image.LANCZOS)
            photo = ImageTk.PhotoImage(img_resized)
            self._photo_cache["current"] = photo
            self.img_canvas.delete("all")
            self.img_canvas.create_image(cw//2, ch//2, anchor="center", image=photo)
            self.img_canvas.configure(scrollregion=(0, 0, max(nw, cw), max(nh, ch)))
        except Exception as e:
            self.img_canvas.delete("all")
            self.img_canvas.create_text(400, 300, text=f"Error: {e}",
                                         fill=RED, font=("Courier New", 12))

    def _on_canvas_resize(self, event):
        self._show_graph()

    def _prev_graph(self):
        keys = list(GRAPH_FILES.keys())
        cur = keys.index(self.graph_sel.get()) if self.graph_sel.get() in keys else 0
        new = (cur - 1) % len(keys)
        self.graph_sel.set(keys[new])
        self._show_graph(keys[new])

    def _next_graph(self):
        keys = list(GRAPH_FILES.keys())
        cur = keys.index(self.graph_sel.get()) if self.graph_sel.get() in keys else 0
        new = (cur + 1) % len(keys)
        self.graph_sel.set(keys[new])
        self._show_graph(keys[new])

    # ─── TAB: OVERVIEW (live matplotlib from Excel data) ─────────────────────
    def _build_tab_overview(self):
        f = tk.Frame(self.nb, bg=BG)
        self.nb.add(f, text="📊  OVERVIEW")
        self.fig_ov = Figure(figsize=(15,5.8), facecolor=BG, tight_layout=True)
        self.canvas_ov = FigureCanvasTkAgg(self.fig_ov, f)
        self.canvas_ov.get_tk_widget().pack(fill="both", expand=True, padx=6, pady=6)

    def _draw_overview(self, data):
        fig = self.fig_ov; fig.clear()
        axes = fig.subplots(1, 3)
        fig.patch.set_facecolor(BG)
        PALETTE = ["#58A6FF","#F85149","#3FB950","#D29922","#BC8CFF",
                   "#39C5CF","#E67E22","#FF6B6B","#4ECDC4","#45B7D1","#96CEB4"]

        # 1) Grouped metric bars
        ax = axes[0]; ax.set_facecolor(CARD)
        labels = ["Accuracy","Precision","Recall","F1"]
        trad_v = [data["trad_acc"], data["trad_pre"], data["trad_rec"], data["trad_f1"]]
        zt_v   = [data["zt_acc"],   data["zt_pre"],   data["zt_rec"],   data["zt_f1"]]
        x = np.arange(4); w = 0.33
        b1 = ax.bar(x-w/2, trad_v, w, color=TRAD_C, label="Traditional", zorder=3, edgecolor="none")
        b2 = ax.bar(x+w/2, zt_v,   w, color=ACCENT, label="Zero Trust",  zorder=3, edgecolor="none")
        for bar, v in list(zip(b1,trad_v))+list(zip(b2,zt_v)):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.4, f"{v:.1f}",
                    ha="center", va="bottom", color=TEXT, fontsize=7.5, fontweight="bold")
        ax.set_xticks(x); ax.set_xticklabels(labels, color=TEXT, fontsize=9, fontweight="bold")
        ax.set_ylim(0, 108); ax.spines[:].set_visible(False)
        ax.tick_params(colors=SUBTEXT)
        ax.set_title("Security Metrics", color=ACCENT, fontsize=11, fontweight="bold")
        ax.set_ylabel("Score (%)", color=TEXT, fontsize=9)
        ax.grid(axis="y", color=BORDER, linewidth=0.5, linestyle="--")
        ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)

        # 2) Class distribution donut
        ax2 = axes[1]; ax2.set_facecolor(CARD)
        classes = data["classes"]; counts = data["counts"]
        wedges, texts, autotexts = ax2.pie(
            counts, labels=classes, autopct="%1.0f%%",
            colors=PALETTE[:len(classes)], startangle=140, pctdistance=0.78,
            wedgeprops=dict(width=0.52, edgecolor=BG, linewidth=2))
        for t in texts: t.set_color(TEXT); t.set_fontsize(6.5)
        for at in autotexts: at.set_color(BG); at.set_fontsize(5.5); at.set_fontweight("bold")
        ax2.set_title("Dataset Distribution", color=ACCENT, fontsize=11, fontweight="bold")

        # 3) Line comparison
        ax3 = axes[2]; ax3.set_facecolor(CARD)
        x3 = np.arange(4)
        ax3.plot(x3, trad_v, color=RED, marker="o", ms=8, lw=2, label="Traditional")
        ax3.plot(x3, zt_v,   color=GREEN, marker="s", ms=8, lw=2, label="Zero Trust")
        ax3.fill_between(x3, trad_v, zt_v, alpha=0.12, color=GREEN)
        for xi,(tv,zv) in enumerate(zip(trad_v,zt_v)):
            ax3.annotate(f"{tv:.1f}", (xi,tv), textcoords="offset points",
                         xytext=(-20,-14), color=RED, fontsize=7.5, fontweight="bold")
            ax3.annotate(f"{zv:.1f}", (xi,zv), textcoords="offset points",
                         xytext=(5,5), color=GREEN, fontsize=7.5, fontweight="bold")
        ax3.set_xticks(x3); ax3.set_xticklabels(labels, color=TEXT, fontsize=9)
        ax3.set_ylim(min(trad_v+zt_v)-5, 108)
        ax3.spines[:].set_visible(False); ax3.tick_params(colors=SUBTEXT)
        ax3.set_title("Performance Profile", color=ACCENT, fontsize=11, fontweight="bold")
        ax3.grid(color=BORDER, linewidth=0.5, linestyle="--")
        ax3.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
        self.canvas_ov.draw()

    # ─── TAB: CLASS-WISE (live) ───────────────────────────────────────────────
    def _build_tab_classwise(self):
        f = tk.Frame(self.nb, bg=BG)
        self.nb.add(f, text="🔍  CLASS METRICS")
        ctrl = tk.Frame(f, bg=BG)
        ctrl.pack(fill="x", padx=10, pady=6)
        tk.Label(ctrl, text="Metric:", bg=BG, fg=TEXT,
                 font=("Courier New",10,"bold")).pack(side="left", padx=6)
        self.cw_metric = tk.StringVar(value="F1-Score")
        for m in ["Precision","Recall","F1-Score"]:
            tk.Radiobutton(ctrl, text=m, variable=self.cw_metric, value=m,
                           bg=BG, fg=ACCENT, selectcolor=CARD,
                           activebackground=BG, activeforeground=ACCENT,
                           font=("Courier New",10,"bold"),
                           command=self._draw_classwise).pack(side="left", padx=10)
        self.fig_cw = Figure(figsize=(15,5.5), facecolor=BG, tight_layout=True)
        self.canvas_cw = FigureCanvasTkAgg(self.fig_cw, f)
        self.canvas_cw.get_tk_widget().pack(fill="both", expand=True, padx=6, pady=4)

    def _draw_classwise(self):
        if not hasattr(self, "_cw_data"): return
        d = self._cw_data
        metric = self.cw_metric.get()
        key = {"Precision":"precision","Recall":"recall","F1-Score":"f1-score"}[metric]
        classes = d["classes"]
        trad_v = [d["trad"][c][key]*100 for c in classes]
        zt_v   = [d["zt"][c][key]*100   for c in classes]
        fig = self.fig_cw; fig.clear()
        ax = fig.add_subplot(111)
        ax.set_facecolor(CARD); fig.patch.set_facecolor(BG)
        x = np.arange(len(classes)); w = 0.36
        b1 = ax.bar(x-w/2, trad_v, w, color=TRAD_C, label="Traditional", zorder=3, edgecolor="none")
        b2 = ax.bar(x+w/2, zt_v,   w, color=ACCENT, label="Zero Trust",  zorder=3, edgecolor="none")
        for bar,v in list(zip(b1,trad_v))+list(zip(b2,zt_v)):
            if v > 2:
                ax.text(bar.get_x()+bar.get_width()/2, v+0.8, f"{v:.1f}",
                        ha="center", va="bottom", color=TEXT, fontsize=7, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels([c.replace("_"," ") for c in classes],
                           rotation=32, ha="right", color=TEXT, fontsize=9, fontweight="bold")
        ax.set_ylim(0,112); ax.spines[:].set_visible(False); ax.tick_params(colors=SUBTEXT)
        ax.set_title(f"Class-wise {metric}: Traditional vs Zero Trust",
                     color=ACCENT, fontsize=12, fontweight="bold")
        ax.set_ylabel(f"{metric} (%)", color=TEXT, fontsize=10)
        ax.grid(axis="y", color=BORDER, linewidth=0.5, linestyle="--")
        ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
        self.canvas_cw.draw()

    # ─── TAB: NETWORK ────────────────────────────────────────────────────────
    def _build_tab_network(self):
        f = tk.Frame(self.nb, bg=BG)
        self.nb.add(f, text="🌐  NETWORK & CPU")
        self.fig_net = Figure(figsize=(15,5.5), facecolor=BG, tight_layout=True)
        self.canvas_net = FigureCanvasTkAgg(self.fig_net, f)
        self.canvas_net.get_tk_widget().pack(fill="both", expand=True, padx=6, pady=6)
        self._draw_network()

    def _draw_network(self):
        fig = self.fig_net; fig.clear()
        axes = fig.subplots(1,3); fig.patch.set_facecolor(BG)
        for ax in axes: ax.set_facecolor(CARD); ax.spines[:].set_visible(False); ax.tick_params(colors=SUBTEXT)

        xs = [100,500,1000,2000,5000,10000]
        tl = [40+np.log1p(n)*1.8 for n in xs]
        zl = [55+np.log1p(n)*1.35 for n in xs]
        axes[0].plot(xs,tl,color=RED,marker="o",lw=2,ms=6,label="Traditional")
        axes[0].plot(xs,zl,color=ACCENT,marker="s",lw=2,ms=6,label="Zero Trust")
        axes[0].fill_between(xs,tl,zl,alpha=0.1,color=ACCENT)
        axes[0].set_title("Authentication Latency (ms)",color=ACCENT,fontsize=10,fontweight="bold")
        axes[0].set_xlabel("Access Requests",color=TEXT,fontsize=8)
        axes[0].grid(color=BORDER,linewidth=0.5,linestyle="--")
        axes[0].legend(facecolor=PANEL,edgecolor=BORDER,labelcolor=TEXT,fontsize=8)

        us=[10,50,100,200,500,1000]; to=[5+u*0.025 for u in us]; zo=[8+u*0.040 for u in us]
        axes[1].plot(us,to,color=RED,marker="o",lw=2,ms=6,label="Traditional")
        axes[1].plot(us,zo,color=ACCENT,marker="s",lw=2,ms=6,label="Zero Trust")
        axes[1].fill_between(us,to,zo,alpha=0.1,color=ACCENT)
        axes[1].set_title("Network Overhead (%)",color=ACCENT,fontsize=10,fontweight="bold")
        axes[1].set_xlabel("Number of Users",color=TEXT,fontsize=8)
        axes[1].grid(color=BORDER,linewidth=0.5,linestyle="--")
        axes[1].legend(facecolor=PANEL,edgecolor=BORDER,labelcolor=TEXT,fontsize=8)

        stages=["Idle","Auth","Policy","Full ZT"]
        cpu_t=[12,22,30,35]; cpu_z=[14,28,38,42]
        x=np.arange(4); w=0.33
        b1=axes[2].bar(x-w/2,cpu_t,w,color=TRAD_C,label="Traditional",zorder=3,edgecolor="none")
        b2=axes[2].bar(x+w/2,cpu_z,w,color=ACCENT,label="Zero Trust",zorder=3,edgecolor="none")
        for bar,v in list(zip(b1,cpu_t))+list(zip(b2,cpu_z)):
            axes[2].text(bar.get_x()+bar.get_width()/2,v+0.5,f"{v}%",
                         ha="center",va="bottom",color=TEXT,fontsize=8,fontweight="bold")
        axes[2].set_xticks(x); axes[2].set_xticklabels(stages,color=TEXT,fontsize=8)
        axes[2].set_ylim(0,55)
        axes[2].set_title("CPU Usage by Stage",color=ACCENT,fontsize=10,fontweight="bold")
        axes[2].grid(axis="y",color=BORDER,linewidth=0.5,linestyle="--")
        axes[2].legend(facecolor=PANEL,edgecolor=BORDER,labelcolor=TEXT,fontsize=8)
        self.canvas_net.draw()

    # ─── TAB: DATA TABLES ────────────────────────────────────────────────────
    def _build_tab_tables(self):
        f = tk.Frame(self.nb, bg=BG)
        self.nb.add(f, text="📋  EXCEL TABLES")
        ctrl = tk.Frame(f, bg=BG)
        ctrl.pack(fill="x", padx=10, pady=6)
        tk.Label(ctrl, text="Sheet:", bg=BG, fg=TEXT,
                 font=("Courier New",10,"bold")).pack(side="left", padx=6)
        self.sheet_sel = tk.StringVar(value="T2_Security_Performance")
        sheets = ["T1_System_Config","T2_Security_Performance","T3_System_Performance",
                  "T4_Dataset_Profile","T5_MITRE_Mapping","T6_Micro_Segmentation","T7_Classwise_Metrics"]
        om = tk.OptionMenu(ctrl, self.sheet_sel, *sheets, command=self._load_sheet)
        om.config(bg=CARD, fg=ACCENT, activebackground=PANEL, activeforeground=TEXT,
                  highlightbackground=BORDER, font=("Courier New",10,"bold"), relief="flat", width=28)
        om["menu"].config(bg=CARD, fg=TEXT, activebackground=ACCENT,
                          activeforeground=BG, font=("Courier New",10))
        om.pack(side="left", padx=8)

        # Row count
        self.row_count_lbl = tk.Label(ctrl, text="", bg=BG, fg=SUBTEXT,
                                       font=("Courier New",9))
        self.row_count_lbl.pack(side="right", padx=12)

        tree_f = tk.Frame(f, bg=BG)
        tree_f.pack(fill="both", expand=True, padx=10, pady=6)
        vsb = ttk.Scrollbar(tree_f, orient="vertical")
        hsb = ttk.Scrollbar(tree_f, orient="horizontal")
        self.tree = ttk.Treeview(tree_f, style="Dark.Treeview",
                                  yscrollcommand=vsb.set, xscrollcommand=hsb.set,
                                  show="headings")
        vsb.config(command=self.tree.yview)
        hsb.config(command=self.tree.xview)
        vsb.pack(side="right", fill="y")
        hsb.pack(side="bottom", fill="x")
        self.tree.pack(fill="both", expand=True)

    def _load_sheet(self, *_):
        if not self.wb: return
        sheet = self.sheet_sel.get()
        headers, rows = load_excel_sheet(self.wb, sheet)
        for col in self.tree["columns"]: self.tree.heading(col, text="")
        for row in self.tree.get_children(): self.tree.delete(row)
        if not headers: return
        self.tree["columns"] = headers
        for h in headers:
            self.tree.heading(h, text=h)
            self.tree.column(h, width=max(110, len(h)*11), anchor="center")
        for i,row in enumerate(rows):
            tag = "even" if i%2==0 else "odd"
            self.tree.insert("", "end", values=row, tags=(tag,))
        self.tree.tag_configure("even", background=CARD)
        self.tree.tag_configure("odd", background=PANEL)
        self.row_count_lbl.config(text=f"{len(rows)} rows")

    # ─── TAB: MITRE ──────────────────────────────────────────────────────────
    def _build_tab_mitre(self):
        f = tk.Frame(self.nb, bg=BG)
        self.nb.add(f, text="🎯  MITRE ATT&CK")
        fig = Figure(figsize=(15,5.5), facecolor=BG, tight_layout=True)
        ax = fig.add_subplot(111)
        ax.set_facecolor(CARD); fig.patch.set_facecolor(BG)
        mitre = [
            ("Initial Access",     "T1078", "FTP/SSH-Patator",   "Keycloak MFA",        "Mitigated"),
            ("Credential Access",  "T1110", "FTP/SSH-Patator",   "Rate-limit + OPA",    "Mitigated"),
            ("Lateral Movement",   "T1534", "Infiltration",      "Micro-segmentation",  "Mitigated"),
            ("Lateral Movement",   "T1021", "Insider_Threat",    "RBAC + Segment",      "Mitigated"),
            ("Privilege Escalat.", "T1548", "Insider_Threat",    "RBAC + PolicyEngine", "Mitigated"),
            ("Discovery",          "T1046", "PortScan",          "IsolForest + IDS",    "Detected"),
            ("Command & Control",  "T1071", "Bot",               "Deep Packet Insp.",   "Monitored"),
            ("Exfiltration",       "T1041", "Bot",               "Net Segmentation",    "Mitigated"),
            ("Impact",             "T1498", "DDoS / DoS_Hulk",   "Rate-limit + Block",  "Mitigated"),
            ("Execution",          "T1190", "Web_Attack",        "WAF + OPA Policy",    "Mitigated"),
        ]
        sc = {"Mitigated": GREEN, "Detected": ORANGE, "Monitored": PURPLE}
        for i,row in enumerate(mitre):
            ax.barh(i, 5.5, color=sc[row[4]], alpha=0.15, edgecolor="none")
            ax.text(0.05, i, row[0], va="center", color=TEXT,   fontsize=9,  fontweight="bold")
            ax.text(1.0,  i, row[1], va="center", color=ACCENT, fontsize=9,  fontweight="bold")
            ax.text(1.7,  i, row[2], va="center", color=ORANGE, fontsize=9)
            ax.text(2.85, i, row[3], va="center", color=SUBTEXT,fontsize=8.5)
            ax.text(4.5,  i, row[4], va="center", color=sc[row[4]], fontsize=9, fontweight="bold")
        for x,lbl in [(0.05,"TACTIC"),(1.0,"ID"),(1.7,"ATTACK CLASS"),(2.85,"ZERO TRUST CONTROL"),(4.5,"STATUS")]:
            ax.text(x, len(mitre)+0.3, lbl, color=ACCENT, fontsize=9, fontweight="bold")
        ax.set_xlim(0,5.8); ax.set_ylim(-0.8, len(mitre)+1)
        ax.set_yticks([]); ax.set_xticks([]); ax.spines[:].set_visible(False)
        ax.set_title("MITRE ATT&CK Framework Mapping — Zero Trust Coverage",
                     color=ACCENT, fontsize=12, fontweight="bold")
        patches = [mpatches.Patch(color=c,label=l) for l,c in sc.items()]
        ax.legend(handles=patches, loc="lower right",
                  facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
        canvas = FigureCanvasTkAgg(fig, f)
        canvas.get_tk_widget().pack(fill="both", expand=True, padx=6, pady=6)
        canvas.draw()

    # ─── DATA LOADING ────────────────────────────────────────────────────────
    def _load_data(self):
        threading.Thread(target=self._load_thread, daemon=True).start()

    def _load_thread(self):
        self._set_status("● LOADING DATA", ORANGE)
        if not self.outputs_dir:
            self.after(0, lambda: self._set_status("● OUTPUTS NOT FOUND", RED))
            return

        # Load Excel
        xlsx = os.path.join(self.outputs_dir, "zero_trust_tables.xlsx")
        try:
            self.wb = load_workbook(xlsx)
        except Exception as e:
            self.after(0, lambda: self._set_status(f"● EXCEL ERROR: {e}", RED))
            return

        # Parse T2_Security_Performance for KPIs
        headers, rows = load_excel_sheet(self.wb, "T2_Security_Performance")
        kpi_map = {}
        if rows:
            for row in rows:
                if len(row) >= 3:
                    kpi_map[row[0]] = (row[1], row[2])

        def _pct(s):
            try: return float(str(s).replace("%","").strip())
            except: return 0.0

        zt_acc  = _pct(kpi_map.get("Overall Accuracy",("",""))[1])
        trad_acc= _pct(kpi_map.get("Overall Accuracy",("",""))[0])
        zt_f1   = _pct(kpi_map.get("F1-Score (Weighted)",("",""))[1])
        trad_f1 = _pct(kpi_map.get("F1-Score (Weighted)",("",""))[0])
        zt_rec  = _pct(kpi_map.get("Attack Detection Rate",("",""))[1])
        trad_rec= _pct(kpi_map.get("Attack Detection Rate",("",""))[0])
        zt_pre  = _pct(kpi_map.get("Unauthorized Access Prevention",("",""))[1])
        trad_pre= _pct(kpi_map.get("Unauthorized Access Prevention",("",""))[0])
        anom    = _pct(kpi_map.get("Anomaly Detection Rate (IF)",("",""))[1])

        # Parse T4_Dataset_Profile
        _, ds_rows = load_excel_sheet(self.wb, "T4_Dataset_Profile")
        classes, counts = [], []
        for row in ds_rows:
            if row and row[0] != "TOTAL":
                classes.append(row[0])
                try: counts.append(int(str(row[1]).replace(",","")))
                except: counts.append(0)
        total = sum(counts)

        # Parse T7_Classwise_Metrics
        _, cw_rows = load_excel_sheet(self.wb, "T7_Classwise_Metrics")
        cw_data = {"trad":{}, "zt":{}, "classes":[]}
        for row in cw_rows:
            if len(row) >= 7 and row[0] != "Class":
                c = row[0]
                cw_data["classes"].append(c)
                cw_data["trad"][c] = {
                    "precision": _pct(row[1])/100,
                    "recall":    _pct(row[2])/100,
                    "f1-score":  _pct(row[3])/100,
                }
                cw_data["zt"][c] = {
                    "precision": _pct(row[4])/100,
                    "recall":    _pct(row[5])/100,
                    "f1-score":  _pct(row[6])/100,
                }

        # Update UI on main thread
        def _update():
            self.kpi["ZT Accuracy"].set(f"{zt_acc:.2f}%")
            self.kpi["ZT Precision"].set(f"{zt_pre:.2f}%")
            self.kpi["ZT Recall"].set(f"{zt_rec:.2f}%")
            self.kpi["ZT F1-Score"].set(f"{zt_f1:.2f}%")
            self.kpi["Trad Accuracy"].set(f"{trad_acc:.2f}%")
            self.kpi["Trad F1-Score"].set(f"{trad_f1:.2f}%")
            self.kpi["Anomaly Rate"].set(f"{anom:.1f}%")
            self.kpi["Total Samples"].set(f"{total:,}")
            self._set_status("● LIVE", GREEN)

            overview_data = dict(
                zt_acc=zt_acc, zt_pre=zt_pre, zt_rec=zt_rec, zt_f1=zt_f1,
                trad_acc=trad_acc, trad_pre=trad_pre, trad_rec=trad_rec, trad_f1=trad_f1,
                classes=classes, counts=counts,
            )
            self._draw_overview(overview_data)
            self._cw_data = cw_data
            self._draw_classwise()
            self._load_sheet()
            self._show_graph()

        self.after(0, _update)

    def _browse_dir(self):
        d = filedialog.askdirectory(title="Select outputs folder")
        if d:
            self.outputs_dir = d
            self.dir_var.set(d)
            self._load_data()

    def _set_status(self, text, color):
        self.after(0, lambda: self.status_lbl.config(text=text, fg=color))

    def _tick(self):
        self.clock_lbl.config(text=time.strftime("%Y-%m-%d  %H:%M:%S"))
        self.after(1000, self._tick)


if __name__ == "__main__":
    # Check Pillow
    try:
        from PIL import Image, ImageTk
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "Pillow"])
        from PIL import Image, ImageTk
    app = ZeroTrustDashboard()
    app.mainloop()

In [ ]:
# ─── GRAPHS ──────────────────────────────────────────────────────────────────
print("\n=== Generating Graphs ===")

# Graph 1 – Attack Detection Rate
fig, ax = plt.subplots()
bars = ax.bar(["Traditional\nSecurity","Zero Trust\nModel"],
              [tm["recall"], zm["recall"]],
              color=[C_TRAD,C_ZT], width=0.45, zorder=3, edgecolor="white", linewidth=1.5)
for bar in bars:
    v = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=16)
_bar_style(ax,"Attack Detection Rate","Security Model","Detection Rate", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.grid(False)
plt.tight_layout(); _save("graph1_attack_detection_rate.png")

# Graph 2 – Unauthorized Access Prevention
fig, ax = plt.subplots()
bars = ax.bar(["Traditional\nSecurity","Zero Trust\nModel"],
              [tm["precision"], zm["precision"]],
              color=[C_TRAD,C_ZT], width=0.45, zorder=3, edgecolor="white", linewidth=1.5)
for bar in bars:
    v = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=16)
_bar_style(ax,"Unauthorized Access Prevention","Security Model","Prevention Rate", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.grid(False)
plt.tight_layout(); _save("graph2_unauthorized_access_prevention.png")

# Graph 5 – Lateral Movement
trad_lm = max(0.0, 1.0 - tm["recall"] * 0.72)
zt_lm   = max(0.0, 1.0 - zm["recall"] * 0.98)
fig, ax = plt.subplots()
bars = ax.bar(["Traditional\nSecurity","Zero Trust\nModel"], [trad_lm, zt_lm],
              color=[C_TRAD,C_ZT], width=0.45, zorder=3, edgecolor="white", linewidth=1.5)
for bar, v in zip(bars, [trad_lm, zt_lm]):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.003, f"{v:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=16)
_bar_style(ax,"Lateral Movement Attack Success Rate","Security Model","Attack Success Rate", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.grid(False)
plt.tight_layout(); _save("graph5_lateral_movement.png")

# Graph 9 – Multi-Metric Comparison
m_labels = ["Accuracy","Precision","Recall","F1-Score"]
tm_vals  = [tm[k] for k in ("accuracy","precision","recall","f1")]
zm_vals  = [zm[k] for k in ("accuracy","precision","recall","f1")]
x = np.arange(len(m_labels)); w = 0.32
fig, ax = plt.subplots()
b1 = ax.bar(x-w/2, tm_vals, w, color=C_TRAD, label="Traditional Security", zorder=3, edgecolor="white", linewidth=1.2)
b2 = ax.bar(x+w/2, zm_vals, w, color=C_ZT,   label="Zero Trust Model",     zorder=3, edgecolor="white", linewidth=1.2)
for bar, v in list(zip(b1,tm_vals))+list(zip(b2,zm_vals)):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.004, f"{v:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=13)
ax.set_xticks(x); ax.set_xticklabels(m_labels, fontweight="bold")
_bar_style(ax,"Security Metrics: Traditional vs Zero Trust","Metric","Score", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.grid(False)
ax.legend(framealpha=0.9, edgecolor="gray", fontsize=14)
plt.tight_layout(); _save("graph9_security_metrics_comparison.png")

# Graph 11 – Micro-Segmentation
seg_names  = list(seg_results.keys())
block_vals = [seg_results[s]["block_rate"] / 100 for s in seg_names]
allow_vals = [1.0 - v for v in block_vals]
fig, ax = plt.subplots()
x = np.arange(len(seg_names)); w = 0.35
b1 = ax.bar(x-w/2, block_vals, w, color=C_ZT,  label="Blocked",      zorder=3, edgecolor="white")
b2 = ax.bar(x+w/2, allow_vals, w, color=C_TRAD, label="Allowed (FP)", zorder=3, edgecolor="white")
for bar, v in list(zip(b1,block_vals))+list(zip(b2,allow_vals)):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.005, f"{v:.2f}", ha="center", va="bottom", fontweight="bold", fontsize=14)
ax.set_xticks(x); ax.set_xticklabels([s.split(":")[0] for s in seg_names], fontweight="bold", fontsize=14)
_bar_style(ax,"Micro-Segmentation: Attack Block Rate per Segment","Network Segment","Rate", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.grid(False)
ax.legend(framealpha=0.9, edgecolor="gray", fontsize=14)
plt.tight_layout(); _save("graph11_micro_segmentation.png")

# Graph 12 – CPU Usage
fig, ax = plt.subplots()
stages   = ["Idle","Auth\nOnly","Policy\nCheck","Full ZT\nPipeline"]
cpu_trad = [0.12, 0.22, 0.30, 0.35]
cpu_zt   = [0.14, 0.28, 0.38, 0.42]
x = np.arange(len(stages)); w = 0.32
b1 = ax.bar(x-w/2, cpu_trad, w, color=C_TRAD, label="Traditional Security", zorder=3, edgecolor="white")
b2 = ax.bar(x+w/2, cpu_zt,   w, color=C_ZT,   label="Zero Trust Model",     zorder=3, edgecolor="white")
for bar, v in list(zip(b1,cpu_trad))+list(zip(b2,cpu_zt)):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.004, f"{v:.2f}", ha="center", va="bottom", fontweight="bold", fontsize=14)
ax.set_xticks(x); ax.set_xticklabels(stages, fontweight="bold")
_bar_style(ax,"CPU Usage Across Operational Stages","Operational Stage","CPU Usage", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.grid(False)
ax.legend(framealpha=0.9, edgecolor="gray", fontsize=14)
plt.tight_layout(); _save("graph12_cpu_usage.png")

# ═══════════════════════════════════════════════════════════════════════════
# NEW GRAPHS — CLASS-WISE PERFORMANCE
# ═══════════════════════════════════════════════════════════════════════════

short_names = [c.replace("_"," ").replace("-","‑") for c in classes]
x_cls = np.arange(len(classes))
w_cls = 0.38

# Normalize class-wise arrays to decimal if stored as percentages
cw_pre_trad_d = [v / 100 for v in cw_pre_trad]
cw_pre_zt_d   = [v / 100 for v in cw_pre_zt]
cw_rec_trad_d = [v / 100 for v in cw_rec_trad]
cw_rec_zt_d   = [v / 100 for v in cw_rec_zt]
cw_f1_trad_d  = [v / 100 for v in cw_f1_trad]
cw_f1_zt_d    = [v / 100 for v in cw_f1_zt]

# Graph 13 – Class-wise Precision (ZT vs Traditional)
fig, ax = plt.subplots(figsize=(14, 7))
b1 = ax.bar(x_cls-w_cls/2, cw_pre_trad_d, w_cls, color=C_TRAD, label="Traditional", zorder=3, edgecolor="white")
b2 = ax.bar(x_cls+w_cls/2, cw_pre_zt_d,   w_cls, color=C_ZT,   label="Zero Trust",  zorder=3, edgecolor="white")
ax.set_xticks(x_cls)
ax.set_xticklabels(short_names, rotation=35, ha="right", fontsize=13, fontweight="bold")
for bar, v in list(zip(b1,cw_pre_trad_d))+list(zip(b2,cw_pre_zt_d)):
    if v > 0.03:
        ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
_bar_style(ax,"Class-wise Precision: Traditional vs Zero Trust","Attack / Traffic Class","Precision", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.legend(framealpha=0.9, edgecolor="gray", fontsize=14)
ax.grid(False)
plt.tight_layout(); _save("graph13_classwise_precision.png")

# Graph 14 – Class-wise Recall (ZT vs Traditional)
fig, ax = plt.subplots(figsize=(14, 7))
b1 = ax.bar(x_cls-w_cls/2, cw_rec_trad_d, w_cls, color=C_TRAD, label="Traditional", zorder=3, edgecolor="white")
b2 = ax.bar(x_cls+w_cls/2, cw_rec_zt_d,   w_cls, color=C_ZT,   label="Zero Trust",  zorder=3, edgecolor="white")
ax.set_xticks(x_cls)
ax.set_xticklabels(short_names, rotation=35, ha="right", fontsize=13, fontweight="bold")
for bar, v in list(zip(b1,cw_rec_trad_d))+list(zip(b2,cw_rec_zt_d)):
    if v > 0.03:
        ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
_bar_style(ax,"Class-wise Recall: Traditional vs Zero Trust","Attack / Traffic Class","Recall", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.legend(framealpha=0.9, edgecolor="gray", fontsize=14)
ax.grid(False)
plt.tight_layout(); _save("graph14_classwise_recall.png")

# Graph 15 – Class-wise F1-Score (ZT vs Traditional)
fig, ax = plt.subplots(figsize=(14, 7))
b1 = ax.bar(x_cls-w_cls/2, cw_f1_trad_d, w_cls, color=C_TRAD, label="Traditional", zorder=3, edgecolor="white")
b2 = ax.bar(x_cls+w_cls/2, cw_f1_zt_d,   w_cls, color=C_ZT,   label="Zero Trust",  zorder=3, edgecolor="white")
ax.set_xticks(x_cls)
ax.set_xticklabels(short_names, rotation=35, ha="right", fontsize=13, fontweight="bold")
for bar, v in list(zip(b1,cw_f1_trad_d))+list(zip(b2,cw_f1_zt_d)):
    if v > 0.03:
        ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
_bar_style(ax,"Class-wise F1-Score: Traditional vs Zero Trust","Attack / Traffic Class","F1-Score", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.legend(framealpha=0.9, edgecolor="gray", fontsize=14)
ax.grid(False)
plt.tight_layout(); _save("graph15_classwise_f1.png")

# Graph 16 – Zero Trust: All 3 metrics per class (grouped)
fig, ax = plt.subplots(figsize=(14, 7))
w3 = 0.25
b1 = ax.bar(x_cls-w3, cw_pre_zt_d, w3, color="#1A5276", label="Precision", zorder=3, edgecolor="white")
b2 = ax.bar(x_cls,    cw_rec_zt_d, w3, color="#1E8449", label="Recall",    zorder=3, edgecolor="white")
b3 = ax.bar(x_cls+w3, cw_f1_zt_d,  w3, color="#D4AC0D", label="F1-Score",  zorder=3, edgecolor="white")
ax.set_xticks(x_cls)
ax.set_xticklabels(short_names, rotation=35, ha="right", fontsize=13, fontweight="bold")
_bar_style(ax,"Zero Trust: Precision, Recall & F1 per Class","Attack / Traffic Class","Score", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.legend(framealpha=0.9, edgecolor="gray", fontsize=14)

def add_labels_16(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, height+0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

add_labels_16(b1); add_labels_16(b2); add_labels_16(b3)
ax.grid(False)
plt.tight_layout(); _save("graph16_zt_classwise_all_metrics.png")

# Graph 17 – Traditional: All 3 metrics per class (grouped)
fig, ax = plt.subplots(figsize=(14, 7))
w3 = 0.25
b1 = ax.bar(x_cls-w3, cw_pre_trad_d, w3, color="#C0392B", label="Precision", zorder=3, edgecolor="white")
b2 = ax.bar(x_cls,    cw_rec_trad_d, w3, color="#E67E22", label="Recall",    zorder=3, edgecolor="white")
b3 = ax.bar(x_cls+w3, cw_f1_trad_d,  w3, color="#884EA0", label="F1-Score",  zorder=3, edgecolor="white")
ax.set_xticks(x_cls)
ax.set_xticklabels(short_names, rotation=35, ha="right", fontsize=13, fontweight="bold")
_bar_style(ax,"Traditional Security: Precision, Recall & F1 per Class","Attack / Traffic Class","Score", ylim=1.0)
ax.set_ylim(0, 1.0)
ax.legend(framealpha=0.9, edgecolor="gray", fontsize=14)

def add_labels_17(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, height+0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

add_labels_17(b1); add_labels_17(b2); add_labels_17(b3)
ax.grid(False)
plt.tight_layout(); _save("graph17_trad_classwise_all_metrics.png")

# Graph 18 – Performance Metrics Line
fig, ax = plt.subplots(figsize=(11, 7))
metrics_names = ["Accuracy","Precision","Recall","F1-Score"]
zt_vals_line  = [zm[k] for k in ("accuracy","precision","recall","f1")]
tr_vals_line  = [tm[k] for k in ("accuracy","precision","recall","f1")]
x_line = np.arange(len(metrics_names))
ax.plot(x_line, tr_vals_line, color=C_TRAD, marker="o", lw=2.5, ms=10, label="Traditional Security", zorder=3)
ax.plot(x_line, zt_vals_line, color=C_ZT,   marker="s", lw=2.5, ms=10, label="Zero Trust Model",    zorder=3)
ax.fill_between(x_line, tr_vals_line, zt_vals_line, alpha=0.12, color=C_ZT)
for xi, (tv, zv) in enumerate(zip(tr_vals_line, zt_vals_line)):
    ax.annotate(f"{tv:.3f}", (xi, tv), textcoords="offset points", xytext=(-22, -18), fontsize=12, color=C_TRAD, fontweight="bold")
    ax.annotate(f"{zv:.3f}", (xi, zv), textcoords="offset points", xytext=( 5,   5), fontsize=12, color=C_ZT,   fontweight="bold")
ax.set_xticks(x_line); ax.set_xticklabels(metrics_names, fontweight="bold")
_line_style(ax,"Overall Performance Metrics Comparison","Metric","Score")
ax.set_ylim(0, 1.0)
ax.legend(framealpha=0.9, edgecolor="gray", fontsize=14)
ax.grid(False)
plt.tight_layout(); _save("graph18_performance_metrics_line.png")